# AI Smart Recycle Bin - Model Training on Google Colab

This notebook trains a custom **TensorFlow / MediaPipe Object Detection Model** for classifying waste materials (**Metal, Paper, Plastic**) on edge devices like the **Raspberry Pi 4 / 5**.

### Pipeline Overview:
1. **Environment Setup & Dependencies**
2. **Dataset Acquisition & Preprocessing** (Garbage Classification Dataset / TrashNet)
3. **Model Training & Fine-Tuning** (MobileNetV2 / EfficientDet-Lite)
4. **Model Evaluation & Confusion Matrix**
5. **Quantization & Export to TensorFlow Lite (`best.tflite`)**

## 1. Install Dependencies

In [ ]:
!pip install -q mediapipe-model-maker tensorflow matplotlib opencv-python

## 2. Import Libraries

In [ ]:
import os
import json
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from google.colab import files

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

## 3. Download & Prepare Dataset
Download the dataset (Metal, Paper, Plastic samples) from public repositories or Google Drive.

In [ ]:
# Download sample Garbage Classification Dataset
!wget -q -O dataset.zip "https://github.com/garythung/trashnet/raw/master/data/dataset-resized.zip"

# Unzip dataset
with zipfile.ZipFile("dataset.zip", 'r') as zip_ref:
    zip_ref.extractall("dataset")

print("Dataset extracted:")
!ls dataset/dataset-resized

## 4. Train Edge AI Object Detector with MediaPipe Model Maker
We use MediaPipe Model Maker with an EfficientDet-Lite0 backbone optimized for Raspberry Pi real-time inference.

In [ ]:
from mediapipe_model_maker import object_detector

# Load dataset using Pascal VOC or COCO format annotations
# Classes: metal, paper, plastic
train_data = object_detector.Dataset.from_pascal_voc_folder(
    data_dir="dataset/train",
    cache_dir="/tmp/cache_train"
)

val_data = object_detector.Dataset.from_pascal_voc_folder(
    data_dir="dataset/val",
    cache_dir="/tmp/cache_val"
)

# Configure Hyperparameters
spec = object_detector.SupportedModels.MOBILENET_V2
hparams = object_detector.HParams(export_dir='exported_model', epochs=30, batch_size=16, learning_rate=0.001)
options = object_detector.ObjectDetectorOptions(
    supported_model=spec,
    hparams=hparams
)

# Train Model
model = object_detector.ObjectDetector.create(
    train_data=train_data,
    validation_data=val_data,
    options=options
)

## 5. Evaluate Model Performance

In [ ]:
metrics = model.evaluate(val_data)
print(f"Validation Mean Average Precision (mAP): {metrics['AP']:.4f}")

## 6. Export Quantized TFLite Model for Raspberry Pi

In [ ]:
# Export model to TFLite format with embedded metadata
model.export_model('best.tflite')

print("Model exported successfully as 'best.tflite'!")
print(f"File size: {os.path.getsize('exported_model/best.tflite') / (1024*1024):.2f} MB")

# Download model to local PC for deployment to Raspberry Pi
files.download('exported_model/best.tflite')